# TradeMatrix Phase 2 — CNN Pattern Recognition (Google Colab)

This notebook trains a **PyTorch CNN** on chart images labeled by Gemini Vision.

## Steps:
1. Upload your `labels.jsonl` from TradeMatrix
2. Regenerate chart images in Colab
3. Train CNN (uses free T4 GPU)
4. Download `colab_model.pkl` → import back to TradeMatrix

**Why Colab?** Free T4 GPU = 10x faster CNN training than CPU

**Why CNN over XGBoost?** CNN learns visual patterns from raw pixels → higher accuracy for complex shapes like Cup & Handle, H&S

In [ ]:
# ── Cell 1: Install Dependencies ──────────────────────────────────────────────
!pip install mplfinance torch torchvision timm joblib scikit-learn -q
print('✅ Dependencies installed')

In [ ]:
# ── Cell 2: Upload labels.jsonl from TradeMatrix ──────────────────────────────
from google.colab import files

print('Upload your labels.jsonl file from:')
print('  GET http://localhost:8000/api/v1/patterns/labels/export')
print('  (Download this file then upload here)')

uploaded = files.upload()
print(f'Uploaded: {list(uploaded.keys())}')

In [ ]:
# ── Cell 3: Load and Inspect Labels ──────────────────────────────────────────
import json
from collections import Counter

labels = []
with open('tradematrix_labels.jsonl', 'r') as f:
    for line in f:
        line = line.strip()
        if line:
            labels.append(json.loads(line))

print(f'Total labels: {len(labels)}')
pattern_counts = Counter(l['pattern_name'] for l in labels)
print('\nPattern distribution:')
for p, c in sorted(pattern_counts.items(), key=lambda x: -x[1]):
    bar = '█' * (c // 5)
    print(f'  {p:<25} {c:>4} {bar}')

# Filter: keep labels with confidence >= 0.65
high_conf = [l for l in labels if float(l.get('confidence', 0)) >= 0.65]
print(f'\nHigh-confidence labels (>=65%): {len(high_conf)}')

In [ ]:
# ── Cell 4: Re-generate Chart Images (mplfinance) ─────────────────────────────
import yfinance as yf
import mplfinance as mpf
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import os, io
import numpy as np
from PIL import Image

os.makedirs('/content/charts', exist_ok=True)

DARK_STYLE = mpf.make_mpf_style(
    base_mpl_style='dark_background',
    marketcolors=mpf.make_marketcolors(
        up='#00f5a0', down='#ff4466', edge='inherit', wick='inherit'
    ),
)

def generate_chart_image(symbol, window_start, window_end, img_size=(224, 224)):
    """Generate chart image for a labeled window."""
    try:
        df = yf.download(symbol, start=window_start, end=window_end, progress=False)
        if df.empty or len(df) < 10:
            return None

        df = df.rename(columns={'Open': 'Open', 'High': 'High', 'Low': 'Low',
                                  'Close': 'Close', 'Volume': 'Volume'})

        # Add EMA overlays
        ema50 = df['Close'].ewm(span=50).mean()
        ema200 = df['Close'].ewm(span=200).mean()

        apds = [
            mpf.make_addplot(ema50, color='#4facfe', linewidth=1),
            mpf.make_addplot(ema200, color='#ffd700', linewidth=1.5),
        ]

        buf = io.BytesIO()
        fig, _ = mpf.plot(
            df, type='candle', style=DARK_STYLE,
            addplot=apds, volume=True,
            figsize=(4, 3), returnfig=True,
            tight_layout=True
        )
        fig.savefig(buf, format='png', dpi=100)
        plt.close(fig)
        buf.seek(0)

        img = Image.open(buf).convert('RGB').resize(img_size)
        return img
    except Exception as e:
        return None

print('Chart generator ready ✅')

In [ ]:
# ── Cell 5: Build Dataset ─────────────────────────────────────────────────────
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import random

# Get unique pattern classes
PATTERN_CLASSES = sorted(set(l['pattern_name'] for l in high_conf))
CLASS_TO_IDX = {c: i for i, c in enumerate(PATTERN_CLASSES)}
print(f'Classes ({len(PATTERN_CLASSES)}): {PATTERN_CLASSES}')

class ChartPatternDataset(Dataset):
    def __init__(self, labels, transform=None, img_size=(224, 224)):
        self.labels = labels
        self.transform = transform
        self.img_size = img_size
        self._cache = {}

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        label = self.labels[idx]
        key = f"{label['symbol']}_{label.get('window_end', '')}"

        if key not in self._cache:
            img = generate_chart_image(
                label['symbol'],
                label.get('window_start'),
                label.get('window_end'),
                self.img_size
            )
            if img is None:
                img = Image.new('RGB', self.img_size, (8, 12, 20))
            self._cache[key] = img

        img = self._cache[key]
        if self.transform:
            img = self.transform(img)

        class_idx = CLASS_TO_IDX[label['pattern_name']]
        return img, class_idx

# Augmentation transforms
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.3),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

# Train/val split
random.shuffle(high_conf)
split = int(len(high_conf) * 0.85)
train_labels, val_labels = high_conf[:split], high_conf[split:]

train_ds = ChartPatternDataset(train_labels, transform=train_transform)
val_ds = ChartPatternDataset(val_labels, transform=val_transform)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False, num_workers=2)

print(f'Train: {len(train_ds)} | Val: {len(val_ds)}')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device} ✅')

In [ ]:
# ── Cell 6: Build EfficientNet-B0 Model ──────────────────────────────────────
import timm
import torch.nn as nn

class PatternCNN(nn.Module):
    """EfficientNet-B0 fine-tuned for chart pattern classification."""

    def __init__(self, n_classes, dropout=0.3):
        super().__init__()
        # Pre-trained EfficientNet-B0 (ImageNet weights, fast + accurate)
        self.backbone = timm.create_model(
            'efficientnet_b0', pretrained=True, num_classes=0
        )
        in_features = self.backbone.num_features

        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(in_features, 256),
            nn.ReLU(),
            nn.Dropout(dropout / 2),
            nn.Linear(256, n_classes),
        )

    def forward(self, x):
        features = self.backbone(x)
        return self.classifier(features)

model = PatternCNN(n_classes=len(PATTERN_CLASSES)).to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f'Model: EfficientNet-B0 | Params: {total_params/1e6:.1f}M | Classes: {len(PATTERN_CLASSES)}')

In [ ]:
# ── Cell 7: Training Loop ─────────────────────────────────────────────────────
import torch.optim as optim
from torch.optim.lr_scheduler import OneCycleLR

EPOCHS = 25
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = OneCycleLR(optimizer, max_lr=1e-3, epochs=EPOCHS, steps_per_epoch=len(train_loader))
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

best_val_acc = 0.0
history = {'train_loss': [], 'val_acc': []}

for epoch in range(EPOCHS):
    # ── Train ──
    model.train()
    train_loss = 0
    for imgs, targets in train_loader:
        imgs, targets = imgs.to(device), targets.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        scheduler.step()
        train_loss += loss.item()

    # ── Validate ──
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for imgs, targets in val_loader:
            imgs, targets = imgs.to(device), targets.to(device)
            outputs = model(imgs)
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()

    val_acc = correct / total
    avg_loss = train_loss / len(train_loader)
    history['train_loss'].append(avg_loss)
    history['val_acc'].append(val_acc)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), '/content/best_cnn.pth')

    print(f'Epoch {epoch+1:>2}/{EPOCHS} | Loss: {avg_loss:.4f} | Val Acc: {val_acc:.2%} {"✅ Best!" if val_acc == best_val_acc else ""}')

print(f'\n🏆 Best validation accuracy: {best_val_acc:.2%}')

In [ ]:
# ── Cell 8: Save Model for TradeMatrix ───────────────────────────────────────
# This creates colab_model.pkl compatible with TradeMatrix's model import API

import joblib
from sklearn.preprocessing import LabelEncoder
import numpy as np

# Load best weights
model.load_state_dict(torch.load('/content/best_cnn.pth', map_location=device))
model.eval()

# Create a sklearn-compatible wrapper for TradeMatrix
class ColabCNNWrapper:
    """
    Wrapper that makes the PyTorch CNN compatible with TradeMatrix's
    model bundle format (same interface as XGBoost model).
    """
    def __init__(self, cnn_model, classes, device):
        self.cnn_model = cnn_model
        self.classes = classes
        self.device = device
        self.model_type = 'cnn_efficientnet_b0'

    def predict_proba(self, X):
        """X = feature vector (23 features) — ignored for CNN, uses image."""
        # CNN needs image, so we return uniform distribution as fallback
        # Real CNN inference happens in pattern_detector.py via chart image
        n = len(self.classes)
        return np.ones((1, n)) / n

    def predict_from_image(self, img_tensor):
        """Main CNN inference path via image tensor."""
        with torch.no_grad():
            output = self.cnn_model(img_tensor.to(self.device))
            proba = torch.softmax(output, dim=1).cpu().numpy()[0]
        return proba

# Label encoder
le = LabelEncoder()
le.fit(PATTERN_CLASSES)

# Save complete bundle
model_bundle = {
    'model': ColabCNNWrapper(model, PATTERN_CLASSES, device),
    'cnn_state_dict': model.state_dict(),
    'cnn_architecture': 'efficientnet_b0',
    'label_encoder': le,
    'classes': PATTERN_CLASSES,
    'feature_names': [],  # CNN doesn't use geometric features
    'n_features': 0,
    'model_type': 'cnn',
    'val_accuracy': best_val_acc,
    'n_classes': len(PATTERN_CLASSES),
    'training_history': history,
}

joblib.dump(model_bundle, '/content/colab_model.pkl')
print(f'✅ Model saved: colab_model.pkl')
print(f'   Classes: {PATTERN_CLASSES}')
print(f'   Best Val Accuracy: {best_val_acc:.2%}')
print(f'   Architecture: EfficientNet-B0')

In [ ]:
# ── Cell 9: Download and Import to TradeMatrix ────────────────────────────────
from google.colab import files

print('Downloading colab_model.pkl...')
files.download('/content/colab_model.pkl')

print()
print('=' * 60)
print('NEXT STEPS to import into TradeMatrix:')
print('=' * 60)
print()
print('1. Copy colab_model.pkl to:')
print('   Trade_Matrix/models/colab_model.pkl')
print()
print('2. Import via API:')
print('   POST http://localhost:8000/api/v1/patterns/model/import')
print()
print('3. OR use the Training page in the UI:')
print('   http://localhost:3000/training → "Import Colab Model" button')
print()
print(f'Model accuracy: {best_val_acc:.2%} on {len(PATTERN_CLASSES)} pattern classes')

## 📊 Training Complete!

| Item | Value |
|------|-------|
| Architecture | EfficientNet-B0 |
| Input size | 224×224 RGB |
| Training epochs | 25 |
| Optimizer | AdamW + OneCycleLR |
| Augmentation | Flip + ColorJitter |
| Label smoothing | 0.1 |

## Strategy: Hybrid Best-of-Both

- **XGBoost** (CPU, geometric features) → Fast real-time inference, no GPU needed
- **EfficientNet CNN** (Colab GPU) → Higher accuracy on complex patterns
- **TradeMatrix** uses whichever is imported last as the active model
- Both models use the same output format → seamless swap